# Payments Risk Rules Engine — Exploration Notebook

This notebook demonstrates how the rules engine scores payment transactions.
We'll explore three things:

1. **How rules fire** — which rules trigger for each scenario
2. **How scores change** — how the risk score shifts as signals stack up
3. **How tiers shift** — mapping scores to actionable risk tiers

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

from risk_engine.engine import RiskEngine
from risk_engine.models import Transaction
from risk_engine.explain import summarise

In [ ]:
config_dir = Path("..") / "config"
engine = RiskEngine(config_dir=config_dir)

## 1. How rules fire

Each sample transaction is designed to isolate a single rule.
Let's score them and inspect the audit trail.

In [ ]:
with open("sample_transactions.json") as f:
    raw_txns = json.load(f)

for raw in raw_txns:
    scenario = raw.get("_scenario", "")
    txn = Transaction(
        **{k: v for k, v in raw.items() if not k.startswith("_")}
    )
    result = engine.score(txn)

    print(f"=== {scenario} ===")
    print(f"  Transaction : {result.transaction_id}")
    print(f"  Score       : {result.risk_score}")
    print(f"  Tier        : {result.risk_tier}")
    print(f"  Rules fired : {result.rule_hits or 'none'}")
    for exp in result.explanations:
        print(f"    -> {exp}")
    print()

## 2. How scores change

Risk signals are additive. Let's start with a clean transaction and
progressively layer on suspicious attributes to watch the score climb.

In [ ]:
stages = [
    {
        "label": "Baseline — clean domestic purchase",
        "overrides": {},
    },
    {
        "label": "+ High amount ($2,500)",
        "overrides": {"amount": 2500.0},
    },
    {
        "label": "+ Geo mismatch (merchant in Nigeria)",
        "overrides": {"amount": 2500.0, "merchant_country": "NG"},
    },
    {
        "label": "+ High velocity (8 recent txns)",
        "overrides": {"amount": 2500.0, "merchant_country": "NG", "recent_transaction_count": 8},
    },
    {
        "label": "+ New device",
        "overrides": {"amount": 2500.0, "merchant_country": "NG", "recent_transaction_count": 8, "known_device": False, "device_id": "DEV-NEW"},
    },
    {
        "label": "+ Nighttime (3 AM)",
        "overrides": {"amount": 2500.0, "merchant_country": "NG", "recent_transaction_count": 8, "known_device": False, "device_id": "DEV-NEW", "hour_of_day": 3},
    },
    {
        "label": "+ Risky merchant (gambling)",
        "overrides": {"amount": 2500.0, "merchant_country": "NG", "recent_transaction_count": 8, "known_device": False, "device_id": "DEV-NEW", "hour_of_day": 3, "merchant_category": "gambling"},
    },
]

base = dict(
    transaction_id="TXN-EXPLORE",
    account_id="A001",
    amount=75.0,
    merchant_country="US",
    account_home_country="US",
    merchant_category="retail",
    device_id="DEV-100",
    hour_of_day=14,
    recent_transaction_count=0,
    known_device=True,
)

print(f"{'Stage':<50} {'Score':>6}  {'Tier':<8} Rules fired")
print("-" * 100)

for stage in stages:
    txn_data = {**base, **stage["overrides"]}
    txn = Transaction(**txn_data)
    result = engine.score(txn)
    hits = ", ".join(result.rule_hits) or "—"
    print(f"{stage['label']:<50} {result.risk_score:>6.2f}  {result.risk_tier:<8} {hits}")

## 3. How tiers shift

Tier boundaries determine the operational response:

| Tier   | Score range  | Action            |
|--------|-------------|-------------------|
| low    | 0.00 – 0.40 | Allow             |
| medium | 0.40 – 0.70 | Flag for review   |
| high   | 0.70 – 0.90 | Step-up auth      |
| block  | 0.90 – 1.00 | Decline           |

Let's craft transactions that land in each tier.

In [ ]:
tier_examples = [
    {
        "label": "Low — routine purchase",
        "txn": Transaction(
            transaction_id="TIER-LOW",
            account_id="A001",
            amount=45.0,
            merchant_country="US",
            account_home_country="US",
        ),
    },
    {
        "label": "Medium — high amount + geo mismatch",
        "txn": Transaction(
            transaction_id="TIER-MED",
            account_id="A001",
            amount=800.0,
            merchant_country="BR",
            account_home_country="US",
        ),
    },
    {
        "label": "High — amount + geo + velocity + nighttime",
        "txn": Transaction(
            transaction_id="TIER-HIGH",
            account_id="A002",
            amount=1500.0,
            merchant_country="NG",
            account_home_country="US",
            hour_of_day=2,
            recent_transaction_count=10,
        ),
    },
    {
        "label": "Block — every signal firing",
        "txn": Transaction(
            transaction_id="TIER-BLOCK",
            account_id="A003",
            amount=5000.0,
            merchant_country="NG",
            account_home_country="US",
            merchant_category="gambling",
            device_id="DEV-NEW",
            hour_of_day=3,
            recent_transaction_count=20,
            known_device=False,
        ),
    },
]

for ex in tier_examples:
    result = engine.score(ex["txn"])
    bar = "\u2588" * int(result.risk_score * 40)
    print(f"\n--- {ex['label']} ---")
    print(f"  Score : {result.risk_score:.2f}  {bar}")
    print(f"  Tier  : {result.risk_tier}")
    print(f"  Hits  : {result.rule_hits}")
    for exp in result.explanations:
        print(f"    -> {exp}")

## Full audit trail

The `details` field on every scoring result includes all six rules —
triggered or not — with their weights. This is the full audit record.

In [ ]:
txn = Transaction(
    transaction_id="AUDIT-001",
    account_id="A001",
    amount=950.0,
    merchant_country="BR",
    account_home_country="US",
    hour_of_day=3,
    recent_transaction_count=1,
    known_device=True,
)

result = engine.score(txn)

print(f"Transaction : {result.transaction_id}")
print(f"Final score : {result.risk_score}")
print(f"Final tier  : {result.risk_tier}")
print()
print(f"{'Rule':<26} {'Fired':<8} {'Weight':>6}  Explanation")
print("-" * 90)
for d in result.details:
    fired = "YES" if d.triggered else "no"
    weight = f"{d.weight:.2f}" if d.triggered else "  —"
    exp = d.explanation or "—"
    print(f"{d.rule_name:<26} {fired:<8} {weight:>6}  {exp}")